In [1]:
import json
import pennylane as qml
import pennylane.numpy as np

# Helper function to build the S(phi) gate
def S(phi):
    return np.array([[np.exp(1j * phi), 0], [0, np.exp(-1j * phi)]], dtype=np.complex128)

# Helper function to build the U(a) gate as defined
def U(a):
    sqrt = np.sqrt(1 - a ** 2)
    return np.array([[a, sqrt], [sqrt, -a]], dtype=np.complex128)

# Function to get QSP matrix for a given a and phase angles
def qsp_matrix(a, angles):
    # angles: [phi0, phi1, phi2, phi3] applied as:
    # S(phi0) U(a) S(phi1) U(a) S(phi2) U(a) S(phi3)
    phi0, phi1, phi2, phi3 = angles
    M = S(phi0) @ U(a) @ S(phi1) @ U(a) @ S(phi2) @ U(a) @ S(phi3)
    return M

def coefficients(angles):
    """ This function returns the coefficients associated with the polynomial generated by 
    the QSP routine as a function of the phase angles.
    Args:
        - angles (np.array(float)): Array of real numbers containing the four phase angles,
        in reverse order of application.
    Returns:       
        - (np.array(complex)): A numpy array containing the coefficients of the polynomial
        generated by QSP, where the first element is the coefficient of the cubic term and
        the second element is for the linear term.
    """

    # Choose values of a for a cubic polynomial (degree 3, odd parity, so only terms a^3 and a)
    # We'll use a = -1, 0, 1, i/√2, -i/√2 to extract both real and imaginary parts robustly.
    # But specifically, since for a degree 3 odd parity polynomial:
    #    P(a) = α a^3 + β a,
    # we can set up two linear equations for two values of a ≠ 0:
    #    P(a1) = α a1^3 + β a1
    #    P(a2) = α a2^3 + β a2
    # Solve for α and β.

    # Use a1 = 1, a2 = -1 (good for real coefficients); for complex case, also use imaginary a
    # For general complex coefficients, better to use 3 points and least squares for robustness.

    # Get P(a) for a = 1, -1, 0, i/√2, -i/√2
    avals = [1.0, -1.0, 0.0, 1j / np.sqrt(2), -1j / np.sqrt(2)]
    # The even parity term is zero, as only a^3 and a survive (odd parity).
    Pa = []
    for a in avals:
        mat = qsp_matrix(a, angles)
        Pa.append(mat[0, 0])
    Pa = np.array(Pa)

    # We want to solve for α, β in: for each a,
    #   P(a) = α * a^3 + β * a
    # For all 5 a's, this defines a linear system:
    #   [a1^3, a1]   [α]   = [P(a1)]
    #   [a2^3, a2] * [β]   = [P(a2)]
    #   ...

    A = np.array([[a**3, a] for a in avals], dtype=np.complex128)
    # Least squares solution in case of complex overdetermined system:
    sol, residuals, rank, s = np.linalg.lstsq(A, Pa, rcond=None)
    # sol contains [α, β]
    return sol

def run(test_case_input: str) -> str:
    ins = np.array(json.loads(test_case_input))
    coeffs = coefficients(ins)
    outs = [[np.real(elem).numpy(), np.imag(elem).numpy()] for elem in coeffs]
    
    return str(outs)

def check(solution_output: str, expected_output: str) -> None:
    solution_output = json.loads(solution_output)
    expected_output = json.loads(expected_output)

    assert np.allclose(solution_output, expected_output, rtol = 1e-4)

test_cases = [
    ('[0.5,0.8,1.0,1.0]', '[[-0.1707976, -2.4084889], [-0.8166822, 2.2507432]]'),
    ('[-0.20409113, -0.91173829, 0.91173829, 0.20409113]', '[[2.5,0],[-1.5,0]]')
]

for i, (input_, expected_output) in enumerate(test_cases):
    print(f"Running test case {i} with input '{input_}'...")

    try:
        output = run(input_)
    except Exception as exc:
        print(f"Runtime Error. {exc}")

    else:
        if message := check(output, expected_output):
            print(f"Wrong Answer. Have: '{output}'. Want: '{expected_output}'.")
        else:
            print("Correct!")

Running test case 0 with input '[0.5,0.8,1.0,1.0]'...
Correct!
Running test case 1 with input '[-0.20409113, -0.91173829, 0.91173829, 0.20409113]'...
Correct!
